In [ ]:
import pandas as pd
import numpy as np

# 1. 加载真实数据（含 E_t）
df_true = pd.read_csv('finance_with_Et.csv')  # 包含 t, y, return, ..., E_t

# 2. 加载预测
df_A = pd.read_csv('pred_A.csv')  # t, y_pred_A
df_B = pd.read_csv('pred_B.csv')  # t, y_pred_B

# 3. 按 t 合并（确保顺序一致）
df = df_true[['t', 'E_t']].copy()
df = df.merge(df_A[['t', 'y_pred_A']], on='t', how='inner')
df = df.merge(df_B[['t', 'y_pred_B']], on='t', how='inner')

print("Merged data shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
L = 10  # 与计算 E_t 时的窗口一致！

# 为 Model A 构造特征
features_A = []
for i in range(len(df)):
    if i < L:
        # 不足 L 步，用 NaN 填充（后续会删除）
        features_A.append([np.nan] * 6)
    else:
        window = df['y_pred_A'].iloc[i-L:i].values  # [t-L, t-1]
        features_A.append([
            np.mean(window),
            np.std(window),
            np.max(np.abs(window)),
            np.mean(np.abs(window)),
            np.min(window),
            np.max(window)
        ])

# 为 Model B 构造特征
features_B = []
for i in range(len(df)):
    if i < L:
        features_B.append([np.nan] * 6)
    else:
        window = df['y_pred_B'].iloc[i-L:i].values
        features_B.append([
            np.mean(window),
            np.std(window),
            np.max(np.abs(window)),
            np.mean(np.abs(window)),
            np.min(window),
            np.max(window)
        ])

# 转为 DataFrame
feat_cols = ['mean', 'std', 'max_abs', 'mean_abs', 'min', 'max']
df_feat_A = pd.DataFrame(features_A, columns=[f'A_{c}' for c in feat_cols])
df_feat_B = pd.DataFrame(features_B, columns=[f'B_{c}' for c in feat_cols])

# 合并回主 DataFrame
df = pd.concat([df, df_feat_A, df_feat_B], axis=1)

# 删除前 L 行（特征为 NaN）
df = df.iloc[L:].reset_index(drop=True)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Model A 的探针数据
X_A = df[[f'A_{c}' for c in feat_cols]].values
y = df['E_t'].values  # 共享标签

# Model B 的探针数据
X_B = df[[f'B_{c}' for c in feat_cols]].values

In [ ]:
def train_probe(X, y, name):
    # 划分（在测试集内部划分，避免过拟合）
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )
    
    # 标准化
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    
    # 训练
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(X_train_sc, y_train)
    
    # 评估
    auc = roc_auc_score(y_test, clf.predict_proba(X_test_sc)[:, 1])
    print(f"{name} Probe AUC: {auc:.4f}")
    return auc, clf, scaler

auc_A, clf_A, scaler_A = train_probe(X_A, y, "Model A")
auc_B, clf_B, scaler_B = train_probe(X_B, y, "Model B")

In [ ]:
from scipy import stats

# 简单方法：bootstrap 估计 AUC 标准差
def bootstrap_auc(X, y, clf, scaler, n_bootstrap=1000):
    aucs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(X), len(X), replace=True)
        X_bs = X[idx]
        y_bs = y[idx]
        X_bs_sc = scaler.transform(X_bs)
        auc = roc_auc_score(y_bs, clf.predict_proba(X_bs_sc)[:, 1])
        aucs.append(auc)
    return np.mean(aucs), np.std(aucs)

mean_A, std_A = bootstrap_auc(X_A, y, clf_A, scaler_A)
mean_B, std_B = bootstrap_auc(X_B, y, clf_B, scaler_B)

print(f"Model A: {mean_A:.4f} ± {std_A:.4f}")
print(f"Model B: {mean_B:.4f} ± {std_B:.4f}")

# 简单 z-test
z = (mean_A - mean_B) / np.sqrt(std_A**2 + std_B**2)
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"p-value: {p_value:.4f}")